In [ ]:
df_model = df.copy()
def convertir_a_lista(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else []
    except:
        return []

df_model['genres'] = df_model['genres'].apply(convertir_a_lista)

cat_cols = ['series', 'language', 'bookFormat']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = df_model[col].astype(str).fillna('missing')
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le

df_model['awards'] = df_model['awards'].astype(bool)
df_model['firstPublishDate'] = df_model['firstPublishDate'].astype(bool)

author_avg = df_model.groupby('author')['rating'].mean()
df_model['author_rating'] = df_model['author'].map(author_avg)
df_model['author_rating'] = df_model['author_rating'].fillna(df_model['rating'].mean())

mlb = MultiLabelBinarizer()
genres_ohe = mlb.fit_transform(df_model['genres'])
df_genres = pd.DataFrame(genres_ohe, columns=mlb.classes_, index=df_model.index)
df_model = pd.concat([df_model, df_genres], axis=1)

features = ['series', 'language', 'bookFormat', 'firstPublishDate', 'awards', 'author_rating'] + list(df_genres.columns)

df_train = df_model[df_model['rating'].notna()]
df_predict = df_model[df_model['rating'].isna()]

X_train = df_train[features]
y_train = df_train['rating']
X_pred = df_predict[features]

for k in [2, 3, 4]:
    print(f"\nImputando con KNN (k={k})...")
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train, y_train)
    rating_pred = knn.predict(X_pred)
    missing_index = df[df['rating'].isna()].index
    df.loc[missing_index, 'rating'] = rating_pred
    print(f"Ratings imputados con K={k}. Nulos restantes: {df['rating'].isna().sum()}")